# Module 05 — Assignment: the residual network's toolkit

Module 04's assignment built what AlexNet needed to train a big network without memorizing.
This one builds what ResNet needed to train a *deep* one. You implement four load-bearing
pieces — **batch normalization's forward pass** (training and evaluation modes), **its
backward pass** (the famous one), **the residual block** with its forked backward, and the
**zero-padding shortcut** — and the check cells grade you against finite differences and the
canonical `python/resnet.py` + `nanograd` answer key. The finale drops your BatchNorm and your
residual block into a GIVEN harness that must reproduce the module's toy result: a 34-layer
plain network that cannot fit two spirals, and its residual twin that fits every point.

Rules: don't peek at `lib/python/nanograd` or `python/resnet.py` until you've tried. The check
cells are your feedback. Fill every `# TODO`, run top to bottom.

In [ ]:
# Setup — run once. (GIVEN: do not modify.)
import os, sys, math
import numpy as np

def _find_repo():
    here = os.getcwd()
    cands = [here] + [os.path.abspath(os.path.join(here, *(['..'] * k))) for k in range(1, 6)]
    for base in cands:
        if os.path.isdir(os.path.join(base, 'lib', 'python', 'nanograd')):
            return base
    return here

REPO = _find_repo()
TOPIC = os.path.join(REPO, 'topics', '05-resnet')
sys.path.insert(0, os.path.join(REPO, 'lib', 'python'))
sys.path.insert(0, os.path.join(TOPIC, 'python'))
sys.path.insert(0, os.path.join(TOPIC, 'tests'))
import nanograd as ng
import resnet as canon
from check_utils import (rel_error, eval_numerical_gradient_array, compare_to_canonical)
from nanograd.rng import Rng
print('setup ok — nanograd and check utils imported')

## Part 1 — batch normalization, forward

For each feature (column) $c$ of a batch $X$ of shape $(m, C)$, normalize with the batch's own
statistics, then rescale with learned $\gamma, \beta$:

$$\mu_c = \frac{1}{m}\sum_i x_{ic}, \qquad \sigma_c^2 = \frac{1}{m}\sum_i (x_{ic} - \mu_c)^2, \qquad
\hat{x}_{ic} = \frac{x_{ic} - \mu_c}{\sqrt{\sigma_c^2 + \epsilon}}, \qquad y_{ic} = \gamma_c \hat{x}_{ic} + \beta_c .$$

While training, also update the running averages that evaluation will use (PyTorch's
convention: `momentum` is the weight of the newest batch, and the running variance is the
*unbiased* one, $\sigma_c^2 \cdot m/(m-1)$):

$$\text{run\_mean} \leftarrow (1-\text{mom})\,\text{run\_mean} + \text{mom}\,\mu, \qquad
\text{run\_var} \leftarrow (1-\text{mom})\,\text{run\_var} + \text{mom}\,\sigma^2\tfrac{m}{m-1}.$$

In evaluation mode, normalize with the running statistics instead (and leave them alone).
Cache `self.xhat` and `self.invstd` $= 1/\sqrt{\cdot + \epsilon}$ — Part 2 needs them. (This
is the dense $(m, C)$ case; the library's version also handles conv maps by treating every
pixel as another sample of its channel.)

In [ ]:
class MyBatchNorm:
    def __init__(self, C, eps=1e-5, momentum=0.1):
        self.gamma, self.beta = np.ones(C), np.zeros(C)          # GIVEN
        self.running_mean, self.running_var = np.zeros(C), np.ones(C)
        self.eps, self.momentum = eps, momentum
        self.training = True

    def forward(self, X):
        m = X.shape[0]
        if self.training:
            ###########################################################################
            # TODO: mu and the BIASED var over axis 0; update self.running_mean
            # and self.running_var as above (unbiased var for the running one);
            # self.invstd = 1/sqrt(var + eps); self.xhat = (X - mu) * invstd. ~6 lines.
            ###########################################################################
            pass
            ###########################################################################
            #                          END OF YOUR CODE
            ###########################################################################
        else:
            ###########################################################################
            # TODO: the same normalization with the running statistics. ~2 lines.
            ###########################################################################
            pass
            ###########################################################################
            #                          END OF YOUR CODE
            ###########################################################################
        return self.gamma * self.xhat + self.beta                # GIVEN

    def backward(self, dY):
        return my_batchnorm_backward(self, dY)                   # GIVEN: Part 2

    def params(self): return [self.gamma, self.beta]
    def grads(self): return [self.dgamma, self.dbeta]

In [ ]:
# Check: your forward equals nanograd's BatchNorm -- outputs and running statistics,
# in training mode, then in evaluation mode with the statistics you accumulated.
rng = np.random.default_rng(0)
you, lib = MyBatchNorm(5), ng.BatchNorm(5)
for bn in (you, lib):
    bn.gamma[:] = [1.0, 2.0, 0.5, -1.0, 3.0]; bn.beta[:] = [0.0, 1.0, -2.0, 0.5, 0.0]
for _ in range(3):
    Xb = rng.standard_normal((16, 5)) * 4 + 2
    assert rel_error(you.forward(Xb), lib.forward(Xb)) < 1e-12, 'training-mode output differs'
print('training-mode mean/var after BN:', np.round(you.xhat.mean(0), 6), np.round(you.xhat.var(0), 4))
assert rel_error(you.running_mean, lib.running_mean) < 1e-12, 'running_mean differs'
assert rel_error(you.running_var, lib.running_var) < 1e-12, 'running_var differs (unbiased? momentum?)'
you.training = lib.training = False
Xe = rng.standard_normal((4, 5))
assert rel_error(you.forward(Xe), lib.forward(Xe)) < 1e-12, 'evaluation-mode output differs'
compare_to_canonical([float(you.running_var.sum()), float(you.forward(Xe).sum())],
                     [float(lib.running_var.sum()), float(lib.forward(Xe).sum())],
                     labels=['rv_sum', 'y_eval'])

### Inline Question 1
(a) A `Linear` layer followed by a BatchNorm is built with `bias=False`. Why is a bias there
useless — what would BatchNorm do to it? (b) Try `MyBatchNorm(3).forward(np.ones((1, 3)) * 5)` in
training mode. What comes out, and why does evaluation mode not have this problem?
*Your answer:* ____

## Part 2 — batch normalization, backward

Every $x_j$ moves every $\hat{x}_i$: directly, and through $\mu$ and $\sigma^2$, which the
whole batch shares. With $g_i = \partial L/\partial y_i$ (per feature column), the chain rule
collapses to

$$\frac{\partial L}{\partial \beta} = \sum_i g_i, \qquad \frac{\partial L}{\partial \gamma} = \sum_i g_i \hat{x}_i, \qquad
\frac{\partial L}{\partial x_j} = \frac{\gamma\,\text{invstd}}{m}\Big(m\,g_j - \frac{\partial L}{\partial \beta} - \hat{x}_j\,\frac{\partial L}{\partial \gamma}\Big).$$

(Derive it yourself first: $\partial \hat{x}_i / \partial x_j = \text{invstd}\,([i=j] - 1/m - \hat{x}_i\hat{x}_j/m)$.)
In evaluation mode the statistics are constants, so the layer is a per-feature affine map and
$\partial L/\partial x = g\,\gamma\,\text{invstd}$. Set `bn.dgamma` and `bn.dbeta` and return $dX$.

In [ ]:
def my_batchnorm_backward(bn, dY):
    m = dY.shape[0]
    dX = None
    ###########################################################################
    # TODO: bn.dbeta and bn.dgamma (sums over axis 0); then dX by the formula
    # above in training mode, or dY * gamma * invstd in evaluation mode
    # (bn.training tells you which). ~5 lines.
    ###########################################################################

    ###########################################################################
    #                          END OF YOUR CODE
    ###########################################################################
    return dX

In [ ]:
# Check 1: finite differences, training and evaluation mode, for dX, dgamma, dbeta.
rng = np.random.default_rng(1)
for training in (True, False):
    bn = MyBatchNorm(4); bn.gamma[:] = rng.standard_normal(4); bn.beta[:] = rng.standard_normal(4)
    bn.running_mean[:] = rng.standard_normal(4); bn.running_var[:] = rng.random(4) + 0.5
    bn.training = training
    Xb = rng.standard_normal((7, 4)) * 2 + 1
    rm, rv = bn.running_mean.copy(), bn.running_var.copy()
    def f(_):
        bn.running_mean, bn.running_var = rm.copy(), rv.copy()     # keep the stats fixed
        return bn.forward(Xb)
    dYb = rng.standard_normal((7, 4)); f(None); dXb = bn.backward(dYb)
    for name, a, wrt in (('dX', dXb, Xb), ('dgamma', bn.dgamma, bn.gamma), ('dbeta', bn.dbeta, bn.beta)):
        f(None); bn.backward(dYb)
        err = rel_error(a, eval_numerical_gradient_array(f, wrt, dYb))
        print(f"{'train' if training else 'eval '} {name:6s} rel_error = {err:.1e}"); assert err < 1e-6

# Check 2: the full m x m Jacobian (the definition), and the projection property.
bn = MyBatchNorm(3); bn.gamma[:] = [0.5, 2.0, -1.0]
Xj = rng.standard_normal((6, 3)); dYj = rng.standard_normal((6, 3)); bn.forward(Xj); dXj = bn.backward(dYj)
assert np.abs(dXj - canon.naive_batchnorm_backward(Xj, bn.gamma, dYj)).max() < 1e-12
print('per feature: sum(dX) =', np.round(dXj.sum(0), 12), '  sum(dX * xhat) =', (dXj * bn.xhat).sum(0))
assert np.abs(dXj.sum(0)).max() < 1e-10, 'each feature\'s input gradient must sum to zero'
lib = ng.BatchNorm(3); lib.gamma[:] = bn.gamma; lib.forward(Xj)
compare_to_canonical([float((dXj * np.arange(18).reshape(6, 3)).sum()), float(bn.dgamma.sum())],
                     [float((lib.backward(dYj) * np.arange(18).reshape(6, 3)).sum()), float(lib.dgamma.sum())],
                     labels=['dX_wsum', 'dgamma'])

### Inline Question 2
Check 2 printed $\sum_j \partial L/\partial x_j = 0$ for every feature. Prove it from the
formula, and explain in one sentence why it *must* be so: what happens to BatchNorm's output
if you add the same constant to every $x_j$ of a feature? Multiplying them all by the same
positive factor should be just as invisible — yet $\sum_j \hat{x}_j\,\partial L/\partial x_j$
came out small but **not** zero. Which term of the forward pass breaks the invariance, and
how does the sum scale with it? *Your answer:* ____

## Part 3 — the residual block

$$z = \mathcal{F}(x) + s(x), \qquad y = \mathrm{relu}(z),$$

where $\mathcal{F}$ is the block's `body` (a list of layers run in order) and $s$ is the
shortcut — the identity when `shortcut` is `None`. Backward: $x$ reached $z$ by two routes,
so its gradient is the **sum** of the gradient through the body and the gradient back
through the shortcut (for the identity, just $\partial L/\partial z$ itself).

In [ ]:
class MyResidual:
    def __init__(self, body, shortcut=None):
        self.body, self.shortcut = list(body), shortcut          # GIVEN

    def forward(self, X):
        out = None
        ###########################################################################
        # TODO: run X through every layer of self.body in order; add the shortcut
        # of X (X itself if self.shortcut is None, else self.shortcut.forward(X));
        # remember the sum as self.Z; out = relu of the sum. ~5 lines.
        ###########################################################################

        ###########################################################################
        #                          END OF YOUR CODE
        ###########################################################################
        return out

    def backward(self, dY):
        dX = None
        ###########################################################################
        # TODO: through the ReLU (gradient only where self.Z > 0); then back
        # through the body layers in REVERSE order; then add the shortcut's
        # gradient (dZ itself for the identity, else self.shortcut.backward(dZ)). ~5 lines.
        ###########################################################################

        ###########################################################################
        #                          END OF YOUR CODE
        ###########################################################################
        return dX

    def params(self): return [p for l in self.body for p in l.params()]
    def grads(self): return [g for l in self.body for g in l.grads()]
    def train(self):
        for l in self.body:
            if hasattr(l, 'training'): l.training = True
    def eval(self):
        for l in self.body:
            if hasattr(l, 'training'): l.training = False

In [ ]:
# Check: identical to nanograd's Residual (same layers, same weights), and finite differences.
rng = np.random.default_rng(2)
def make_body():
    r = Rng(5); l1, l2 = ng.Linear(6, 6, bias=False), ng.Linear(6, 6, bias=False)
    ng.he_normal(l1.W, r); ng.he_normal(l2.W, r)
    return [l1, ng.BatchNorm(6), ng.ReLU(), l2, ng.BatchNorm(6)]
mine, lib = MyResidual(make_body()), ng.Residual(make_body())
Xr = rng.standard_normal((9, 6)); dYr = rng.standard_normal((9, 6))
Y_you, Y_lib = mine.forward(Xr), lib.forward(Xr)
dX_you, dX_lib = mine.backward(dYr), lib.backward(dYr)
print('forward identical:', np.array_equal(Y_you, Y_lib), '  backward identical:', np.array_equal(dX_you, dX_lib))
assert rel_error(Y_you, Y_lib) < 1e-12 and rel_error(dX_you, dX_lib) < 1e-12
err = rel_error(dX_you, eval_numerical_gradient_array(lambda _: mine.forward(Xr), Xr, dYr))
print(f'dX vs finite differences: rel_error = {err:.1e}'); assert err < 1e-6
compare_to_canonical([float(dX_you.sum()), float(mine.body[0].dW.sum())],
                     [float(dX_lib.sum()), float(lib.body.layers[0].dW.sum())], labels=['dX_sum', 'dW1_sum'])

### Inline Question 3
Drop the ReLU after the addition (the "pre-activation" block, $x_{\ell+1} = x_\ell + \mathcal{F}(x_\ell)$)
and stack 50 blocks. Write $\partial L/\partial x_1$ in terms of $\partial L/\partial x_{51}$. Which
term guarantees that block 1 receives a gradient however badly blocks 2–50 behave — and what is
the corresponding expression for a 50-block *plain* network? *Your answer:* ____

## Part 4 — the zero-padding shortcut (option A)

Where a block halves the resolution (stride 2) and widens from `in_ch` to `out_ch` channels,
the identity no longer fits. The paper's option (A) keeps the shortcut parameter-free: take
every `stride`-th pixel, $X[:, :, ::S, ::S]$, and place those `in_ch` channels at offset
$(\text{out\_ch} - \text{in\_ch})/2$ among `out_ch` channels whose other planes are zero. The
backward pass is the same selection in reverse; the zero planes were not inputs.

In [ ]:
class MyZeroPadShortcut:
    def __init__(self, in_ch, out_ch, stride=2):
        self.in_ch, self.out_ch, self.stride = in_ch, out_ch, stride   # GIVEN
        self.lo = (out_ch - in_ch) // 2

    def forward(self, X):
        self.x_shape = X.shape
        Y = None
        ###########################################################################
        # TODO: subsample X every self.stride pixels in both spatial axes; build a
        # zero array of shape (n, out_ch, OH, OW); copy the subsample into channels
        # lo .. lo+in_ch. ~4 lines.
        ###########################################################################

        ###########################################################################
        #                          END OF YOUR CODE
        ###########################################################################
        return Y

    def backward(self, dY):
        dX = None
        ###########################################################################
        # TODO: zeros of self.x_shape; the kept pixels get dY's channels
        # lo .. lo+in_ch back. ~2 lines.
        ###########################################################################

        ###########################################################################
        #                          END OF YOUR CODE
        ###########################################################################
        return dX

    def params(self): return []
    def grads(self): return []

In [ ]:
# Check: identical to nanograd's ZeroPadShortcut, plus finite differences -- then inside a
# downsampling conv residual block (YOUR block, YOUR shortcut) against the library's.
rng = np.random.default_rng(3)
Xs = rng.standard_normal((2, 3, 7, 7))
you, lib = MyZeroPadShortcut(3, 6), ng.ZeroPadShortcut(3, 6)
Ys = you.forward(Xs); print('output shape', Ys.shape)
assert Ys.shape == (2, 6, 4, 4) and np.array_equal(Ys, lib.forward(Xs))
dYs = rng.standard_normal(Ys.shape)
assert np.array_equal(you.backward(dYs), lib.backward(dYs))
err = rel_error(you.backward(dYs), eval_numerical_gradient_array(lambda _: you.forward(Xs), Xs, dYs))
print(f'dX vs finite differences: rel_error = {err:.1e}'); assert err < 1e-6

ref = canon.build_test_block(Rng(canon.BLOCK_SEED))                 # nanograd's block + GAP
r_blk = ref.layers[0]
mine = MyResidual(r_blk.body.layers, MyZeroPadShortcut(2, 4, 2))     # same layers, your wiring
Xk = rng.standard_normal((3, 2, 6, 6)); dYk = rng.standard_normal((3, 4))
Y_ref = ref.forward(Xk); dX_ref = ref.backward(dYk)
gap = ng.GlobalAvgPool2D()
Y_you = gap.forward(mine.forward(Xk)); dX_you = mine.backward(gap.backward(dYk))
compare_to_canonical([float(Y_you.sum()), float(dX_you.sum()), float((dX_you ** 2).sum())],
                     [float(Y_ref.sum()), float(dX_ref.sum()), float((dX_ref ** 2).sum())],
                     labels=['y_sum', 'dX_sum', 'dX_sq'])

### Inline Question 4
On CIFAR-10 the paper used option (A) everywhere, so that "our residual models have exactly
the same depth, width, and number of parameters as the plain counterparts". On ImageNet it
compared (A) with (B), a strided 1×1 convolution on the shortcuts that change shape. Why
does (A) make the plain-versus-residual comparison cleaner? And why did (B) do slightly
*better* on ImageNet? (Hint: what do the zero channels learn?) *Your answer:* ____

## Verify: your pieces reproduce the module's toy

A GIVEN harness builds the 34-layer toy network — stem, sixteen blocks, head — from the
library's `Linear` layers and **your** `MyBatchNorm`, wiring each block as **your**
`MyResidual` (the residual twin) or as a plain chain (the plain twin), from the same initial
weights as `run_toy()`. Both train for 400 full-batch SGD steps. Your residual twin must fit
every spiral point, as the canonical run does; your plain twin must fail to.

In [ ]:
def train_toy_with(bn_cls, residual_cls, residual, steps=canon.TOY_STEPS):
    '''GIVEN harness: the toy twins trained with YOUR pieces. Do not modify.'''
    X, y = canon.make_spirals()
    r = Rng(canon.TOY_SEED)
    stem = ng.Linear(2, canon.WIDTH); ng.he_normal(stem.W, r)
    blocks = []
    for _ in range(canon.DEPTH):
        l1, l2 = ng.Linear(canon.WIDTH, canon.WIDTH, bias=False), ng.Linear(canon.WIDTH, canon.WIDTH, bias=False)
        ng.he_normal(l1.W, r); ng.he_normal(l2.W, r)
        body = [l1, bn_cls(canon.WIDTH), ng.ReLU(), l2, bn_cls(canon.WIDTH)]
        blocks.append(residual_cls(body) if residual else ng.Sequential(body + [ng.ReLU()]))
    head = ng.Linear(canon.WIDTH, 2); ng.he_normal(head.W, r)
    layers = [stem, ng.ReLU()] + blocks + [head]
    params = lambda: [p for l in layers for p in l.params()]
    grads = lambda: [g for l in layers for g in l.grads()]
    def run(Xin, train):
        for b in blocks:
            (b.train() if train else b.eval()) if hasattr(b, 'train') else None
        a = Xin
        for l in layers:
            a = l.forward(a)
        return a
    loss = ng.SoftmaxCrossEntropy(); opt = ng.SGD(lr=canon.TOY_LR, momentum=canon.TOY_MOM)
    for _ in range(steps):
        final_loss = loss.forward(run(X, True), y)
        d = loss.backward()
        for l in reversed(layers):
            d = l.backward(d)
        opt.step(params(), grads())
    return float((run(X, False).argmax(1) == y).mean()), final_loss

acc_res, loss_res = train_toy_with(MyBatchNorm, MyResidual, residual=True)
acc_plain, loss_plain = train_toy_with(MyBatchNorm, MyResidual, residual=False)
print(f'trained with YOUR pieces, 34 layers:  plain twin acc {acc_plain:.3f} (loss {loss_plain:.2e})   '
      f'residual twin acc {acc_res:.3f} (loss {loss_res:.2e})')
# The plain twin's training is chaotic (the notebook shows why), so its exact accuracy
# depends on the order your code happens to sum in; its loss stays far above the residual
# twin's whatever that order is.
assert loss_plain > 100 * loss_res, 'the plain twin should fail to fit where the residual twin succeeds'
canon_fp = dict(canon.run_toy(residual=True))
compare_to_canonical([acc_res], [canon_fp['acc']], labels=['res_acc'])
print('OK — BatchNorm, the residual block and the shortcut all pull together: depth works again.')